In [1]:
import pathlib as pl
import pickle as pck
import os

import pandas as pd

cfg_nb = pl.Path("../../load-config.ipynb").resolve(strict=True)
%run $cfg_nb

_NB_SESSION = __session__
NB_NAME = pl.Path(_NB_SESSION).name
NB_PATH = pl.Path(_NB_SESSION).parent
NB_REL_PATH = NB_PATH.relative_to(CONFIG["project_repo"]).joinpath(NB_NAME)

DATA_ROOT = CONFIG["local_hilbert_prefix"].joinpath(CONFIG["hilbert_extracted_seq_workdir"])
assert DATA_ROOT.is_dir()

cache_file_path = prep_cache_file(NB_REL_PATH, "verkko_sex_chrom_short_names.pkl")
if not cache_file_path.is_file():

    toplevel_glob = DATA_ROOT.joinpath("seq_short_names").glob("**/*.fasta")

    fasta_files = []
    for fasta_file in toplevel_glob:
        file_size = os.stat(fasta_file).st_size
        if file_size == 0:
            continue
        fasta_files.append(fasta_file)

    reflevel_glob = DATA_ROOT.joinpath("inject-ref").resolve()
    if not reflevel_glob.is_dir():
        print("warning: ref injection folder does not exist")
    else:
        reflevel_glob = reflevel_glob.glob("**/*.fasta")
        for fasta_file in reflevel_glob:
            fasta_files.append(fasta_file)
    
    cache_dump = {
        "fasta_files": fasta_files
    }
    with open(cache_file_path, "wb") as cache:
        _ = pck.dump(cache_dump, cache)
else:
    with open(cache_file_path, "rb") as cache:
        cache_dump = pck.load(cache)
        fasta_files = cache_dump["fasta_files"]

sample_sheet = []
for fasta_file in fasta_files:
    sample = fasta_file.name.rsplit(".", 1)[0]
    remote_path = replace_path_prefix(fasta_file)
    if "-XX" in sample:
        sex = "female"
    else:
        sex = "male"
    sample_sheet.append((sample, sex, remote_path))

sample_sheet = pd.DataFrame.from_records(
    sample_sheet,
    columns=["sample", "sample_sex", "input_path"]
)
sample_sheet.set_index("sample", inplace=True)

sample_sheet_file = CONFIG["project_repo"].joinpath("samples", "verkko_sex_chrom.tsv")
sample_sheet_file.parent.mkdir(exist_ok=True, parents=True)

with open(sample_sheet_file, "w") as table:
    _ = table.write(f"# {TIMESTAMP}\n")
    _ = table.write(f"# N={sample_sheet.shape[0]}\n")
    sample_sheet.to_csv(table, sep="\t", header=True, index=True)


for select_sex in ["Y", "X"]:
    
    sample_sheet_file = CONFIG["project_repo"].joinpath("samples", f"verkko_sex_chrom.{select_sex}.tsv")
    sample_sheet_file.parent.mkdir(exist_ok=True, parents=True)

    samples = sample_sheet.index.str.endswith(select_sex)
    
    subset = sample_sheet.loc[samples, :].copy()
    with open(sample_sheet_file, "w") as table:
        _ = table.write(f"# {TIMESTAMP}\n")
        _ = table.write(f"# N={subset.shape[0]}\n")
        subset.to_csv(table, sep="\t", header=True, index=True)
    
